# AE101 Sales ETL Project Walkthrough Notebook (Snowflake)
| 저장소: [Sales-ETL-Pipeline-AE101](https://github.com/dongchanlim/Sales-ETL-Pipeline-AE101)
 
## 전체 구조 (Structure)
    - RAW(불변) -> CORE(정제) -> MART(스타 스키마) + SECURITY + AUDIT
    - 대리키(surrogate key) + Unknown 멤버(-1)
    - SCD Type 2 (valid_from / valid_to / is_current)
    - FACT 테이블: fact_sales_line / fact_payment / fact_order_lifecycle
    - is_revenue_eligible 플래그로 매출 판정 여부 결정
    - 참조 오류는 -1 로 적재(FLAGGED), 판단 불가 행만 격리(QUARANTINED)
    - 전체 달력 (Dim_date) 2020~2030 
    - 소프트 삭제 적용(is_deleted )
    - 이메일 마스킹 + 지역 기반 행 수준 보안(Row Access Policy)


## 데이터 계층 (Layer)
    - RAW   : 원본 CSV를 변형 없이 "배치(load_id)" 단위로 계속 쌓음. 지우지 않음.
    - CORE  : 최신 배치를 정제·검증한 결과. 품질 판정(주문 단위)도 여기서 함.
    - MART  : 분석용 스타 스키마(차원 5 + Fact 3) + 리포트 뷰
    - AUDIT : 실행 기록, 데이터 이슈, 품질 검사 결과
    - SECURITY : 마스킹 정책, 행 접근 정책, 역할-지역 매핑

## 데이터 실행 순서 (Execution)
    1. **STEP 0**:  준비                 
    2. **STEP 1**:  DDL 생성            
    3. **STEP 2**:  RUN_CTX 시작(load_id)   
    4. **STEP 3**:  RAW 적재(추가만)     
    5. **STEP 4**:  CORE 정제            
    6. **STEP 5**:  CORE 주문 품질 판정 
    7. **STEP 6**:  감사 기록
    8. **STEP 7**:  [셀 분리] SCD2 차원 -> Fact MERGE -> 품질 검사 -> COMMIT
    9. **STEP 8**:  리포트 뷰 생성
    10. **STEP 9**:  보안: 마스킹 + 행 수준 보안 + 역할
    11. **STEP 10**: 결과 확인
    12. **STEP 11**: SCD2 체험 고객 지역이 바뀌면?
    13. **STEP 12**: 정리

 ## 선행 조건 (Prerequisite)
    - Git 저장소 객체를 SNOWFLAKE에 연결합니다 (**AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE**)
    - main 브랜치의 data/raw 원본 데이터를 파이프라인에 연결합니다.
    - 각 데이터 품질에 대한 docs/data_generation.md 에 정리해두었습니다.
    - 노트북 설정값(Query warehouse)에서 사용하고자 하는 웨어하우스를 지정하세요 (**SNOWFLAKE_LEARNING_WH**)
    - STEP 9 의 마스킹/행 접근 정책은 Snowflake Enterprise Edition 이상에서만 동작합니다.
    - 파이프라인(STEP 2~8)은 ACCOUNTADMIN 또는 SYSADMIN 으로 실행하세요.
     (행 접근 정책을 생성한 뒤, 다른 역할(ROLE)로 실행하면 일부 고객이 보이지 않게 됩습니다.)

> **노트북 사용법**:
> 웨어하우스는 노트북 설정(오른쪽 위 ⚙ Notebook settings → Query warehouse)에서 고르거나
> STEP 0 의 `USE WAREHOUSE` 를 사용하세요.
> 세션 변수(`$load_id`)는 같은 노트북 세션 안에서만 유지되므로 STEP 2~8 은 연속해서 실행하세요.

## STEP 0. 준비

In [ ]:
USE ROLE ACCOUNTADMIN;


CREATE DATABASE IF NOT EXISTS ae101_db;
CREATE SCHEMA IF NOT EXISTS ae101_db.git_repos;
USE SCHEMA ae101_db.git_repos;


CREATE OR REPLACE API INTEGRATION github_api_integration
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/dongchanlim')
  ENABLED = TRUE;


CREATE OR REPLACE GIT REPOSITORY ae101_db.git_repos.sales_etl_pipeline
  API_INTEGRATION = github_api_integration
  ORIGIN = 'https://github.com/dongchanlim/Sales-ETL-Pipeline-AE101.git';


ALTER GIT REPOSITORY ae101_db.git_repos.sales_etl_pipeline FETCH;


SHOW GIT BRANCHES IN GIT REPOSITORY ae101_db.git_repos.sales_etl_pipeline;
LS @ae101_db.git_repos.sales_etl_pipeline/branches/main/;

In [ ]:
-- ACCOUNTADMIN 사용
USE ROLE ACCOUNTADMIN;
-- Git 리포지토리의 최신 변경 사항을 원격(Remote)에서 가져옴(Fetch)
ALTER GIT REPOSITORY AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE FETCH;

### 연결된 웨어하우스 확인
노트북 설정의 Query warehouse 가 지정돼 있으면 건너뛰어도 됩니다. 
이름이 필요하면 아래 셀로 확인하세요. 

| **`AE101_DB` 는 데이터베이스 이름이라 `USE WAREHOUSE` 에 쓸 수 없습니다.**

In [ ]:
SHOW WAREHOUSES;


In [ ]:
SELECT CURRENT_WAREHOUSE() AS current_wh, CURRENT_ROLE() AS role;
-- 비어 있으면 위 SHOW WAREHOUSES 결과의 name 으로 아래 줄 주석을 풀어 실행하세요.
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;

## STEP 1. DDL 
| 처음 실행할 때 한번만 실행합니다.

In [ ]:
-- 데이터베이스 AE101_SALES_ENT 지정 및 사용
CREATE DATABASE IF NOT EXISTS AE101_SALES_ENT;
USE DATABASE AE101_SALES_ENT;

-- 5개의 스키마 생성 
CREATE SCHEMA IF NOT EXISTS RAW;
CREATE SCHEMA IF NOT EXISTS CORE;
CREATE SCHEMA IF NOT EXISTS MART;
CREATE SCHEMA IF NOT EXISTS AUDIT;
CREATE SCHEMA IF NOT EXISTS SECURITY;

-- (1) RAW:

-- (1-1) CSV 파일을 올바르게 파싱하기 위해 CSV 파일 포맷 지정
CREATE OR REPLACE FILE FORMAT RAW.CSV_FORMAT
  TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"' ENCODING = 'UTF8';

-- (1-2) RAW 테이블 원본 + 메타데이터 컬럼(load_id, loaded_at, src_file, src_row)을 추가하여 append할 수 있도록 구성
CREATE TABLE IF NOT EXISTS RAW.CUSTOMERS (
  load_id VARCHAR, loaded_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
  customer_id VARCHAR, customer_name VARCHAR, email VARCHAR, region VARCHAR);

CREATE TABLE IF NOT EXISTS RAW.PRODUCTS (
  load_id VARCHAR, loaded_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
  product_id VARCHAR, product_name VARCHAR, category VARCHAR);

CREATE TABLE IF NOT EXISTS RAW.ORDERS (
  load_id VARCHAR, loaded_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
  order_id VARCHAR, customer_id VARCHAR, order_date VARCHAR, order_status VARCHAR, order_amount VARCHAR);

CREATE TABLE IF NOT EXISTS RAW.ORDER_ITEMS (
  load_id VARCHAR, loaded_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
  order_item_id VARCHAR, order_id VARCHAR, product_id VARCHAR, quantity VARCHAR, unit_price VARCHAR);

CREATE TABLE IF NOT EXISTS RAW.PAYMENTS (
  load_id VARCHAR, loaded_at TIMESTAMP_NTZ, src_file VARCHAR, src_row NUMBER,
  payment_id VARCHAR, order_id VARCHAR, payment_status VARCHAR, payment_amount VARCHAR, paid_at VARCHAR);

-- (1-3)RAW_V_*_LATEST: 테이블별 "최신 배치"만 보여주는 뷰를 설정하여 CORE으로 적재할 때는 해당 뷰를 사용
CREATE OR REPLACE VIEW RAW.V_CUSTOMERS_LATEST   AS SELECT * FROM RAW.CUSTOMERS   QUALIFY loaded_at = MAX(loaded_at) OVER ();
CREATE OR REPLACE VIEW RAW.V_PRODUCTS_LATEST    AS SELECT * FROM RAW.PRODUCTS    QUALIFY loaded_at = MAX(loaded_at) OVER ();
CREATE OR REPLACE VIEW RAW.V_ORDERS_LATEST      AS SELECT * FROM RAW.ORDERS      QUALIFY loaded_at = MAX(loaded_at) OVER ();
CREATE OR REPLACE VIEW RAW.V_ORDER_ITEMS_LATEST AS SELECT * FROM RAW.ORDER_ITEMS QUALIFY loaded_at = MAX(loaded_at) OVER ();
CREATE OR REPLACE VIEW RAW.V_PAYMENTS_LATEST    AS SELECT * FROM RAW.PAYMENTS    QUALIFY loaded_at = MAX(loaded_at) OVER ();

-- (2) AUDIT:

-- (2-1) 전체 파이프라인 메타데이터 확인 용도의 테이블 구성
CREATE TABLE IF NOT EXISTS AUDIT.PIPELINE_RUNS
  load_id VARCHAR NOT NULL, status VARCHAR NOT NULL, started_at TIMESTAMP_NTZ NOT NULL, finished_at TIMESTAMP_NTZ,
  input_rows NUMBER, duplicate_rows NUMBER, quarantined_rows NUMBER, flagged_issues NUMBER,
  revenue_orders NUMBER, revenue_amount NUMBER(38,0), message VARCHAR);

-- (2-2)각 배치당 데이터 이슈 격리 및 마킹 (REMOVED(완전 중복 제거) / QUARANTINED(격리, 미적재) / FLAGGED(적재하되 표시))
CREATE TABLE IF NOT EXISTS AUDIT.DATA_ISSUES (
  load_id VARCHAR NOT NULL, source_file VARCHAR, source_row_number NUMBER, record_key VARCHAR,
  error_code VARCHAR, error_detail VARCHAR, order_id VARCHAR,
  action VARCHAR);   

-- (2-3)각 배치당 데이터 품질 확인 (CORE -> MART 적재 시)
CREATE TABLE IF NOT EXISTS AUDIT.DQ_RESULTS (
  load_id VARCHAR NOT NULL, check_name VARCHAR NOT NULL, failures NUMBER NOT NULL);

-- (3) MART:

-- (3-1) MART DIM 차원에서 사용하는 대리키 설정 (SCD Type 2 용도)
CREATE SEQUENCE IF NOT EXISTS MART.SEQ_CUSTOMER_KEY START = 1 INCREMENT = 1;
CREATE SEQUENCE IF NOT EXISTS MART.SEQ_PRODUCT_KEY  START = 1 INCREMENT = 1;

-- (3-2) DIM_CUSTOMER
CREATE TABLE IF NOT EXISTS MART.DIM_CUSTOMER (
  customer_key   NUMBER DEFAULT MART.SEQ_CUSTOMER_KEY.NEXTVAL NOT NULL,
  customer_id    VARCHAR NOT NULL,
  customer_name  VARCHAR,
  email          VARCHAR,
  region         VARCHAR,
  valid_from     TIMESTAMP_NTZ NOT NULL,
  valid_to       TIMESTAMP_NTZ,
  is_current     BOOLEAN NOT NULL,
  hash_diff      VARCHAR,
  load_id        VARCHAR,
  CONSTRAINT PK_DIM_CUSTOMER PRIMARY KEY (customer_key));

-- (3-3) DIM_PRODUCT
CREATE TABLE IF NOT EXISTS MART.DIM_PRODUCT (
  product_key    NUMBER DEFAULT MART.SEQ_PRODUCT_KEY.NEXTVAL NOT NULL,
  product_id     VARCHAR NOT NULL,
  product_name   VARCHAR,
  category       VARCHAR,
  valid_from     TIMESTAMP_NTZ NOT NULL,
  valid_to       TIMESTAMP_NTZ,
  is_current     BOOLEAN NOT NULL,
  hash_diff      VARCHAR,
  load_id        VARCHAR,
  CONSTRAINT PK_DIM_PRODUCT PRIMARY KEY (product_key));

-- (3-4) Unknown (-1) 설정: 참조 대상이 없는 CUSTOMER/PRODUCT도 NULL 대신 'Unknown'으로 매핑
INSERT INTO MART.DIM_CUSTOMER (customer_key, customer_id, customer_name, email, region, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT -1, '-1', 'Unknown', NULL, 'Unknown', '1900-01-01'::TIMESTAMP_NTZ, NULL, TRUE, NULL, 'SEED'
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_CUSTOMER WHERE customer_key = -1);

INSERT INTO MART.DIM_PRODUCT (product_key, product_id, product_name, category, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT -1, '-1', 'Unknown', 'Unknown', '1900-01-01'::TIMESTAMP_NTZ, NULL, TRUE, NULL, 'SEED'
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_PRODUCT WHERE product_key = -1);

-- (3-5) DIM_DATE: 2020-01-01부터 2030-12-31까지 설정 + Unknown(-1)
CREATE OR REPLACE TABLE MART.DIM_DATE AS
WITH days AS (
  SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, '2020-01-01'::DATE) AS d
  FROM TABLE(GENERATOR(ROWCOUNT => 4018)))
SELECT YEAR(d) * 10000 + MONTH(d) * 100 + DAY(d)       AS date_key,
       d                                               AS calendar_date,
       YEAR(d)                                         AS year,
       QUARTER(d)                                      AS quarter,
       MONTH(d)                                        AS month,
       TO_CHAR(d, 'YYYY-MM')                           AS year_month,
       DAY(d)                                          AS day_of_month,
       DAYOFWEEKISO(d)                                 AS day_of_week,     -- 1=월 ... 7=일
       DAYNAME(d)                                      AS day_name,
       DAYOFWEEKISO(d) IN (6, 7)                       AS is_weekend
FROM days
UNION ALL
SELECT -1, '1900-01-01'::DATE, 1900, 1, 1, 'Unknown', 1, 1, 'Unk', FALSE;

-- (3-6) DIM_ORDER_STATUS
CREATE OR REPLACE TABLE MART.DIM_ORDER_STATUS AS
SELECT * FROM (VALUES
  ( 1, 'completed', '완료', TRUE),
  ( 2, 'pending',   '대기', FALSE),
  ( 3, 'cancelled', '취소', TRUE),
  (-1, 'unknown',   '알 수 없음', FALSE)
) AS v(order_status_key, status_code, status_name, is_final);

-- (3-7) DIM_PAYMENT_STATUS
CREATE OR REPLACE TABLE MART.DIM_PAYMENT_STATUS AS
SELECT * FROM (VALUES
  ( 1, 'succeeded', '성공', TRUE),
  ( 2, 'failed',    '실패', FALSE),
  ( 3, 'pending',   '대기', FALSE),
  (-1, 'unknown',   '알 수 없음', FALSE)
) AS v(payment_status_key, status_code, status_name, is_success);

-- (3-8) FACT_SALES_LINE
-- GRAIN: 주문 상세 1행 Fact
CREATE TABLE IF NOT EXISTS MART.FACT_SALES_LINE (
  order_item_id       VARCHAR NOT NULL,      -- degenerate key (입도 식별자)
  order_id            VARCHAR NOT NULL,      -- degenerate dimension
  customer_key        NUMBER  NOT NULL,      -- 주문일 시점의 고객 버전
  product_key         NUMBER  NOT NULL,      -- 주문일 시점의 상품 버전
  order_date_key      NUMBER  NOT NULL,
  order_status_key    NUMBER  NOT NULL,
  order_date          DATE,
  quantity            NUMBER(38,0),
  unit_price          NUMBER(38,0),
  line_amount         NUMBER(38,0),
  is_revenue_eligible BOOLEAN NOT NULL,      -- 매출 인식 여부 (STEP 5 규칙)
  exclusion_reason    VARCHAR,               -- 매출 제외 사유
  is_deleted          BOOLEAN NOT NULL,      -- 소프트 삭제
  load_id             VARCHAR,
  updated_at          TIMESTAMP_NTZ,
  CONSTRAINT PK_FACT_SALES_LINE PRIMARY KEY (order_item_id),
  CONSTRAINT FK_SL_CUSTOMER FOREIGN KEY (customer_key) REFERENCES MART.DIM_CUSTOMER (customer_key),
  CONSTRAINT FK_SL_PRODUCT  FOREIGN KEY (product_key)  REFERENCES MART.DIM_PRODUCT (product_key));

-- (3-9) FACT_PAYMENT
-- GRAIN: 결제 시도 1건
CREATE TABLE IF NOT EXISTS MART.FACT_PAYMENT (
  payment_id          VARCHAR NOT NULL,
  order_id            VARCHAR,
  customer_key        NUMBER  NOT NULL,
  paid_date_key       NUMBER  NOT NULL,
  payment_status_key  NUMBER  NOT NULL,
  paid_at             TIMESTAMP_NTZ,
  payment_amount      NUMBER(38,0),
  has_valid_order     BOOLEAN NOT NULL,
  is_deleted          BOOLEAN NOT NULL,
  load_id             VARCHAR,
  updated_at          TIMESTAMP_NTZ,
  CONSTRAINT PK_FACT_PAYMENT PRIMARY KEY (payment_id));

-- (3-10) FACT_ORDER_LIFECYCLE
-- GRAIN: 주문 1건 (Accumulating Snapshot). 주문의 진행 단계를 한 행에 누적
CREATE TABLE IF NOT EXISTS MART.FACT_ORDER_LIFECYCLE (
  order_id                     VARCHAR NOT NULL,
  customer_key                 NUMBER  NOT NULL,
  order_date_key               NUMBER  NOT NULL,
  first_success_paid_date_key  NUMBER  NOT NULL,
  order_status_key             NUMBER  NOT NULL,
  order_date                   DATE,
  first_success_paid_at        TIMESTAMP_NTZ,
  order_amount                 NUMBER(38,0),
  line_count                   NUMBER,
  line_amount_total            NUMBER(38,0),
  success_payment_count        NUMBER,
  success_payment_amount       NUMBER(38,0),
  days_to_first_success_payment NUMBER,
  is_revenue_eligible          BOOLEAN NOT NULL,
  exclusion_reason             VARCHAR,
  is_deleted                   BOOLEAN NOT NULL,
  load_id                      VARCHAR,
  updated_at                   TIMESTAMP_NTZ,
  CONSTRAINT PK_FACT_ORDER_LIFECYCLE PRIMARY KEY (order_id));

## STEP 2. 배치 실행 시작하기
| 각 배치 ID(load_id)와 기준 시각(run_ts)을 먼저 발급합니다.

In [ ]:
-- RUN_CTX를 통해 배치마다 구분할 수 있는 메타데이터 값 load_id를 생성합니다.
CREATE OR REPLACE TABLE CORE.RUN_CTX AS
  SELECT UUID_STRING() AS load_id, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS run_ts;
SET load_id = (SELECT load_id FROM CORE.RUN_CTX);
SET run_ts  = (SELECT run_ts  FROM CORE.RUN_CTX);

INSERT INTO AUDIT.PIPELINE_RUNS (load_id, status, started_at) SELECT $load_id, 'RUNNING', $run_ts;
SELECT * FROM CORE.RUN_CTX;

## STEP 3. RAW 적재
| Git 저장소의 CSV 파일들을 사용하여 RAW로 적재하는 "새로운 배치"를 실행합니다. 이 때 기존 배치 기록은 지우지 않습니다.


**주의:**
```
같은 실행(load_id)에서 STEP 3 을 두 번 실행하면 같은 배치에 행이 두 번 들어갑니다.
따라서 새로운 배치를 실행하려면 STEP 2 부터 실행하세요.
```

In [ ]:
-- RAW의 테이블들에 load_id, load_at, src_file, scr_row를 함께 기록하는 테이블을 생성합니다.
-- RAW에서 설정한 CSV_FILEFOMAT과 AE101_DB로 연결한 원본 데이터를 사용합니다.

INSERT INTO RAW.CUSTOMERS (load_id, loaded_at, src_file, src_row, customer_id, customer_name, email, region)
SELECT $load_id, $run_ts, 'customers.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3, $4
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/customers.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

INSERT INTO RAW.PRODUCTS (load_id, loaded_at, src_file, src_row, product_id, product_name, category)
SELECT $load_id, $run_ts, 'products.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/products.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

INSERT INTO RAW.ORDERS (load_id, loaded_at, src_file, src_row, order_id, customer_id, order_date, order_status, order_amount)
SELECT $load_id, $run_ts, 'orders.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3, $4, $5
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/orders.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

INSERT INTO RAW.ORDER_ITEMS (load_id, loaded_at, src_file, src_row, order_item_id, order_id, product_id, quantity, unit_price)
SELECT $load_id, $run_ts, 'order_items.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3, $4, $5
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/order_items.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

INSERT INTO RAW.PAYMENTS (load_id, loaded_at, src_file, src_row, payment_id, order_id, payment_status, payment_amount, paid_at)
SELECT $load_id, $run_ts, 'payments.csv', ROW_NUMBER() OVER (ORDER BY METADATA$FILE_ROW_NUMBER) + 1, $1, $2, $3, $4, $5
FROM @AE101_DB.GIT_REPOS.SALES_ETL_PIPELINE/branches/main/data/raw/payments.csv (FILE_FORMAT => 'AE101_SALES_ENT.RAW.CSV_FORMAT');

In [ ]:
-- 확인: 배치별 행 수
WITH latest_batch AS (
  SELECT MAX(loaded_at) AS max_loaded_at FROM (
    SELECT MAX(loaded_at) AS loaded_at FROM RAW.CUSTOMERS
    UNION ALL SELECT MAX(loaded_at) FROM RAW.PRODUCTS
    UNION ALL SELECT MAX(loaded_at) FROM RAW.ORDERS
    UNION ALL SELECT MAX(loaded_at) FROM RAW.ORDER_ITEMS
    UNION ALL SELECT MAX(loaded_at) FROM RAW.PAYMENTS
  )
)
SELECT 'customers' AS file, load_id, COUNT(*) AS row_count FROM RAW.CUSTOMERS WHERE loaded_at = (SELECT max_loaded_at FROM latest_batch) GROUP BY load_id
UNION ALL SELECT 'products',    load_id, COUNT(*) FROM RAW.PRODUCTS    WHERE loaded_at = (SELECT max_loaded_at FROM latest_batch) GROUP BY load_id
UNION ALL SELECT 'orders',      load_id, COUNT(*) FROM RAW.ORDERS      WHERE loaded_at = (SELECT max_loaded_at FROM latest_batch) GROUP BY load_id
UNION ALL SELECT 'order_items', load_id, COUNT(*) FROM RAW.ORDER_ITEMS WHERE loaded_at = (SELECT max_loaded_at FROM latest_batch) GROUP BY load_id
UNION ALL SELECT 'payments',    load_id, COUNT(*) FROM RAW.PAYMENTS    WHERE loaded_at = (SELECT max_loaded_at FROM latest_batch) GROUP BY load_id;

## STEP 4. CORE 규칙 적용
| 테이블마다 같은 규칙을 적용하고, 각 행에 dq_status 를 붙입니다.

```
DUPLICATE : 완전히 같은 행 -> 1건만 남김 (action = REMOVED)
CONFLICT  : 같은 키에 서로 다른 값 매핑 (action = QUARANTINED)
INVALID   : 필수값 누락 / 숫자·날짜·상태 형식 오류 (action = QUARANTINED)
OK        : CORE 테이블로 이동
```

- 없는 고객/상품을 가리키는 행이 있다면 일단 적재하고 키를 -1(Unknown)로 연결 (FLAGGED)
- 주문 없는 결제가 있다면 일단 적재하고 has_valid_order = FALSE (FLAGGED)
- 주문 없는 주문 상세 내역이 있다면 날짜·고객을 알 수 없어 격리 (QUARANTINED)

In [ ]:
-- ISSUE_WORK 테이블은 각 error_code를 통해 데이터 품질을 따로 체크
CREATE OR REPLACE TABLE CORE.ISSUES_WORK (
  source_file VARCHAR, source_row_number NUMBER, record_key VARCHAR,
  error_code VARCHAR, error_detail VARCHAR, order_id VARCHAR, action VARCHAR);

-- customers
-- 완전히 동일한 비즈니스 키와 값을 가진 행은 제거 (DUPLICATE)
-- 필수값(customer_id, customer_name)이 누락된 행은 격리 (INVALID)
-- 동일한 customer_id를 가졌으나 다른 속성값이 달라 충돌이 발생하는 케이스 감지 (CONFLICT)

CREATE OR REPLACE TABLE CORE.CUSTOMERS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.customer_id), '')   AS customer_id,
         NULLIF(TRIM(r.customer_name), '') AS customer_name,
         NULLIF(TRIM(r.email), '')         AS email,
         NULLIF(TRIM(r.region), '')        AS region,
         ROW_NUMBER() OVER (PARTITION BY r.customer_id, r.customer_name, r.email, r.region ORDER BY r.src_row) AS rn
  FROM RAW.V_CUSTOMERS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.customer_id) AS key_cnt,
         CASE WHEN d.customer_id   IS NULL THEN 'customer_id: required value missing'
              WHEN d.customer_name IS NULL THEN 'customer_name: required value missing' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, customer_id, customer_name, email, region, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, customer_id, customer_name, email, region,
       CASE WHEN customer_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'customers.csv', src_row, customer_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       NULL, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.CUSTOMERS_CHK WHERE dq_status <> 'OK';

CREATE OR REPLACE TABLE CORE.CUSTOMERS AS
SELECT customer_id, customer_name, email, COALESCE(region, 'Unknown') AS region,
       MD5(COALESCE(customer_name, '') || '|' || COALESCE(email, '') || '|' || COALESCE(region, 'Unknown')) AS hash_diff,
       src_row
FROM CORE.CUSTOMERS_CHK WHERE dq_status = 'OK';

--products
-- 완전히 동일한 비즈니스 키와 값을 가진 행은 제거 (DUPLICATE)
-- 필수값(product_id, product_name)이 누락된 행은 격리 (INVALID)
-- 동일한 product_id를 가졌으나 다른 속성값이 달라 충돌이 발생하는 케이스 감지 (CONFLICT)
CREATE OR REPLACE TABLE CORE.PRODUCTS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.product_id), '')   AS product_id,
         NULLIF(TRIM(r.product_name), '') AS product_name,
         NULLIF(TRIM(r.category), '')     AS category,
         ROW_NUMBER() OVER (PARTITION BY r.product_id, r.product_name, r.category ORDER BY r.src_row) AS rn
  FROM RAW.V_PRODUCTS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.product_id) AS key_cnt,
         CASE WHEN d.product_id   IS NULL THEN 'product_id: required value missing'
              WHEN d.product_name IS NULL THEN 'product_name: required value missing'
              WHEN d.category     IS NULL THEN 'category: required value missing' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, product_id, product_name, category, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, product_id, product_name, category,
       CASE WHEN product_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'products.csv', src_row, product_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       NULL, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.PRODUCTS_CHK WHERE dq_status <> 'OK';

CREATE OR REPLACE TABLE CORE.PRODUCTS AS
SELECT product_id, product_name, category,
       MD5(COALESCE(product_name, '') || '|' || COALESCE(category, '')) AS hash_diff,
       src_row
FROM CORE.PRODUCTS_CHK WHERE dq_status = 'OK';

-- orders
--
CREATE OR REPLACE TABLE CORE.ORDERS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.order_id), '')     AS order_id,
         NULLIF(TRIM(r.customer_id), '')  AS customer_id,
         NULLIF(TRIM(r.order_date), '')   AS order_date,
         NULLIF(TRIM(r.order_status), '') AS order_status,
         NULLIF(TRIM(r.order_amount), '') AS order_amount,
         ROW_NUMBER() OVER (PARTITION BY r.order_id, r.customer_id, r.order_date, r.order_status, r.order_amount ORDER BY r.src_row) AS rn
  FROM RAW.V_ORDERS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.order_id) AS key_cnt,
         CASE WHEN d.order_id     IS NULL THEN 'order_id: required value missing'
              WHEN d.customer_id  IS NULL THEN 'customer_id: required value missing'
              WHEN d.order_date   IS NULL THEN 'order_date: required value missing'
              WHEN d.order_status IS NULL THEN 'order_status: required value missing'
              WHEN d.order_amount IS NULL THEN 'order_amount: required value missing'
              WHEN NOT REGEXP_LIKE(d.order_amount, '[0-9]+') THEN 'order_amount: expected non-negative integer KRW/quantity'
              WHEN TRY_TO_NUMBER(d.order_amount) > 1000000000000 THEN 'order_amount: exceeds project input limit'
              WHEN TRY_TO_DATE(d.order_date, 'YYYY-MM-DD') IS NULL THEN 'order_date: invalid date'
              WHEN d.order_status NOT IN ('completed', 'pending', 'cancelled') THEN 'unknown order_status' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, order_id, customer_id, order_date, order_status, order_amount, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, order_id, customer_id, order_date, order_status, order_amount,
       CASE WHEN order_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'orders.csv', src_row, order_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       order_id, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.ORDERS_CHK WHERE dq_status <> 'OK';

-- orders
CREATE OR REPLACE TABLE CORE.ORDERS AS
SELECT c.order_id, c.customer_id, TO_DATE(c.order_date, 'YYYY-MM-DD') AS order_date, c.order_status,
       TO_NUMBER(c.order_amount) AS order_amount,
       c.customer_id IN (SELECT customer_id FROM CORE.CUSTOMERS) AS has_valid_customer,
       c.src_row
FROM CORE.ORDERS_CHK c WHERE c.dq_status = 'OK';

INSERT INTO CORE.ISSUES_WORK
SELECT 'orders.csv', src_row, order_id, 'INVALID_REFERENCE', 'Unknown customer: loaded with customer_key = -1', order_id, 'FLAGGED'
FROM CORE.ORDERS WHERE NOT has_valid_customer;

-- order_items
CREATE OR REPLACE TABLE CORE.ORDER_ITEMS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.order_item_id), '') AS order_item_id,
         NULLIF(TRIM(r.order_id), '')      AS order_id,
         NULLIF(TRIM(r.product_id), '')    AS product_id,
         NULLIF(TRIM(r.quantity), '')      AS quantity,
         NULLIF(TRIM(r.unit_price), '')    AS unit_price,
         ROW_NUMBER() OVER (PARTITION BY r.order_item_id, r.order_id, r.product_id, r.quantity, r.unit_price ORDER BY r.src_row) AS rn
  FROM RAW.V_ORDER_ITEMS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.order_item_id) AS key_cnt,
         CASE WHEN d.order_item_id IS NULL THEN 'order_item_id: required value missing'
              WHEN d.order_id      IS NULL THEN 'order_id: required value missing'
              WHEN d.product_id    IS NULL THEN 'product_id: required value missing'
              WHEN d.quantity      IS NULL THEN 'quantity: required value missing'
              WHEN d.unit_price    IS NULL THEN 'unit_price: required value missing'
              WHEN NOT REGEXP_LIKE(d.quantity, '[0-9]+') THEN 'quantity: expected non-negative integer KRW/quantity'
              WHEN TRY_TO_NUMBER(d.quantity) > 1000000000000 THEN 'quantity: exceeds project input limit'
              WHEN NOT REGEXP_LIKE(d.unit_price, '[0-9]+') THEN 'unit_price: expected non-negative integer KRW/quantity'
              WHEN TRY_TO_NUMBER(d.unit_price) > 1000000000000 THEN 'unit_price: exceeds project input limit'
              WHEN TRY_TO_NUMBER(d.quantity) = 0 THEN 'quantity must be positive' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, order_item_id, order_id, product_id, quantity, unit_price, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, order_item_id, order_id, product_id, quantity, unit_price,
       CASE WHEN order_item_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'order_items.csv', src_row, order_item_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       order_id, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.ORDER_ITEMS_CHK WHERE dq_status <> 'OK';

INSERT INTO CORE.ISSUES_WORK
SELECT 'order_items.csv', src_row, order_item_id, 'INVALID_REFERENCE', 'Parent order not found: quarantined', order_id, 'QUARANTINED'
FROM CORE.ORDER_ITEMS_CHK
WHERE dq_status = 'OK' AND order_id NOT IN (SELECT order_id FROM CORE.ORDERS);

CREATE OR REPLACE TABLE CORE.ORDER_ITEMS AS
SELECT c.order_item_id, c.order_id, c.product_id,
       TO_NUMBER(c.quantity) AS quantity, TO_NUMBER(c.unit_price) AS unit_price,
       c.product_id IN (SELECT product_id FROM CORE.PRODUCTS) AS has_valid_product,
       c.src_row
FROM CORE.ORDER_ITEMS_CHK c
WHERE c.dq_status = 'OK' AND c.order_id IN (SELECT order_id FROM CORE.ORDERS);

INSERT INTO CORE.ISSUES_WORK
SELECT 'order_items.csv', src_row, order_item_id, 'INVALID_REFERENCE', 'Unknown product: loaded with product_key = -1', order_id, 'FLAGGED'
FROM CORE.ORDER_ITEMS WHERE NOT has_valid_product;

-- payments
CREATE OR REPLACE TABLE CORE.PAYMENTS_CHK AS
WITH d AS (
  SELECT r.src_row,
         NULLIF(TRIM(r.payment_id), '')     AS payment_id,
         NULLIF(TRIM(r.order_id), '')       AS order_id,
         NULLIF(TRIM(r.payment_status), '') AS payment_status,
         NULLIF(TRIM(r.payment_amount), '') AS payment_amount,
         NULLIF(TRIM(r.paid_at), '')        AS paid_at,
         ROW_NUMBER() OVER (PARTITION BY r.payment_id, r.order_id, r.payment_status, r.payment_amount, r.paid_at ORDER BY r.src_row) AS rn
  FROM RAW.V_PAYMENTS_LATEST r),
u AS (
  SELECT d.*, COUNT(*) OVER (PARTITION BY d.payment_id) AS key_cnt,
         CASE WHEN d.payment_id     IS NULL THEN 'payment_id: required value missing'
              WHEN d.order_id       IS NULL THEN 'order_id: required value missing'
              WHEN d.payment_status IS NULL THEN 'payment_status: required value missing'
              WHEN d.payment_amount IS NULL THEN 'payment_amount: required value missing'
              WHEN d.paid_at        IS NULL THEN 'paid_at: required value missing'
              WHEN NOT REGEXP_LIKE(d.payment_amount, '[0-9]+') THEN 'payment_amount: expected non-negative integer KRW/quantity'
              WHEN TRY_TO_NUMBER(d.payment_amount) > 1000000000000 THEN 'payment_amount: exceeds project input limit'
              WHEN d.payment_status NOT IN ('succeeded', 'failed', 'pending') THEN 'unknown payment_status'
              WHEN TRY_TO_TIMESTAMP_NTZ(d.paid_at, 'YYYY-MM-DD HH24:MI:SS') IS NULL THEN 'paid_at: invalid timestamp' END AS err
  FROM d WHERE d.rn = 1)
SELECT src_row, payment_id, order_id, payment_status, payment_amount, paid_at, 'DUPLICATE' AS dq_status, CAST(NULL AS VARCHAR) AS err FROM d WHERE rn > 1
UNION ALL
SELECT src_row, payment_id, order_id, payment_status, payment_amount, paid_at,
       CASE WHEN payment_id IS NOT NULL AND key_cnt > 1 THEN 'CONFLICT' WHEN err IS NOT NULL THEN 'INVALID' ELSE 'OK' END, err
FROM u;

INSERT INTO CORE.ISSUES_WORK
SELECT 'payments.csv', src_row, payment_id,
       CASE dq_status WHEN 'DUPLICATE' THEN 'EXACT_DUPLICATE' WHEN 'CONFLICT' THEN 'KEY_CONFLICT' ELSE 'INVALID_VALUE' END,
       CASE dq_status WHEN 'DUPLICATE' THEN 'Identical source row removed' WHEN 'CONFLICT' THEN 'Same business key has conflicting source rows' ELSE err END,
       order_id, CASE dq_status WHEN 'DUPLICATE' THEN 'REMOVED' ELSE 'QUARANTINED' END
FROM CORE.PAYMENTS_CHK WHERE dq_status <> 'OK';

-- payments
CREATE OR REPLACE TABLE CORE.PAYMENTS AS
SELECT c.payment_id, c.order_id, c.payment_status,
       TO_NUMBER(c.payment_amount) AS payment_amount,
       TO_TIMESTAMP_NTZ(c.paid_at, 'YYYY-MM-DD HH24:MI:SS') AS paid_at,
       c.order_id IN (SELECT order_id FROM CORE.ORDERS) AS has_valid_order,
       c.src_row
FROM CORE.PAYMENTS_CHK c WHERE c.dq_status = 'OK';

INSERT INTO CORE.ISSUES_WORK
SELECT 'payments.csv', src_row, payment_id, 'INVALID_REFERENCE', 'Parent order not found: loaded with customer_key = -1', order_id, 'FLAGGED'
FROM CORE.PAYMENTS WHERE NOT has_valid_order;

-- ISSUES_WORK 이슈 타입 확인
SELECT action, error_code, error_detail, COUNT(*) AS events 
FROM CORE.ISSUES_WORK 
GROUP BY action, error_code, error_detail 
ORDER BY action, error_code, error_detail;

## STEP 5. CORE 주문 품질 체크
| 매출 인식 규칙은 달라진 점은 "삭제"가 아니라 "플래그"라는 것.

```
매출 인식 = completed + 관련 행 모두 정상 + 성공 결제 정확히 1건 + 주문금액 = 상세합계 = 결제금액
   exclusion_reason : DEPENDENCY_ERROR / NOT_COMPLETED / SUCCESS_PAYMENT_COUNT / MISSING_ITEMS / AMOUNT_MISMATCH
```

In [ ]:
-- ORDER_QUALITY 
CREATE OR REPLACE TABLE CORE.ORDER_QUALITY AS
WITH blocked AS (
  SELECT DISTINCT order_id FROM CORE.ISSUES_WORK
  WHERE action IN ('QUARANTINED', 'FLAGGED') AND order_id IS NOT NULL),
items AS (
  SELECT order_id, COUNT(*) AS line_count, SUM(quantity * unit_price) AS line_amount_total
  FROM CORE.ORDER_ITEMS GROUP BY order_id),
pays AS (
  SELECT order_id,
         SUM(CASE WHEN payment_status = 'succeeded' THEN 1 ELSE 0 END)              AS success_payment_count,
         SUM(CASE WHEN payment_status = 'succeeded' THEN payment_amount ELSE 0 END) AS success_payment_amount,
         MIN(CASE WHEN payment_status = 'succeeded' THEN paid_at END)               AS first_success_paid_at
  FROM CORE.PAYMENTS WHERE has_valid_order GROUP BY order_id),
q AS (
  SELECT o.order_id, o.customer_id, o.order_date, o.order_status, o.order_amount,
         COALESCE(i.line_count, 0)             AS line_count,
         COALESCE(i.line_amount_total, 0)      AS line_amount_total,
         COALESCE(p.success_payment_count, 0)  AS success_payment_count,
         COALESCE(p.success_payment_amount, 0) AS success_payment_amount,
         p.first_success_paid_at,
         CASE
           WHEN b.order_id IS NOT NULL                    THEN 'DEPENDENCY_ERROR'
           WHEN o.order_status <> 'completed'             THEN 'NOT_COMPLETED'
           WHEN COALESCE(p.success_payment_count, 0) <> 1 THEN 'SUCCESS_PAYMENT_COUNT'
           WHEN COALESCE(i.line_count, 0) = 0             THEN 'MISSING_ITEMS'
           WHEN NOT (o.order_amount = i.line_amount_total AND i.line_amount_total = p.success_payment_amount) THEN 'AMOUNT_MISMATCH'
         END AS exclusion_reason
  FROM CORE.ORDERS o
  LEFT JOIN blocked b ON o.order_id = b.order_id
  LEFT JOIN items   i ON o.order_id = i.order_id
  LEFT JOIN pays    p ON o.order_id = p.order_id)
SELECT q.*, q.exclusion_reason IS NULL AS is_revenue_eligible FROM q;

INSERT INTO CORE.ISSUES_WORK
SELECT 'orders.csv', o.src_row, q.order_id, q.exclusion_reason, 'Order excluded from revenue (still loaded)', q.order_id, 'FLAGGED'
FROM CORE.ORDER_QUALITY q JOIN CORE.ORDERS o ON q.order_id = o.order_id
WHERE q.exclusion_reason IN ('SUCCESS_PAYMENT_COUNT', 'MISSING_ITEMS', 'AMOUNT_MISMATCH');

-- SALES_LINE_PREP: Fact 적재 전 PREP 테이블 
CREATE OR REPLACE TABLE CORE.SALES_LINE_PREP AS
SELECT i.order_item_id, i.order_id, q.customer_id, i.product_id, q.order_date, q.order_status,
       i.quantity, i.unit_price, i.quantity * i.unit_price AS line_amount,
       q.is_revenue_eligible, q.exclusion_reason
FROM CORE.ORDER_ITEMS i JOIN CORE.ORDER_QUALITY q ON i.order_id = q.order_id;

-- PAYMENT_PREP: Fact 적재 전 PREP 테이블 
CREATE OR REPLACE TABLE CORE.PAYMENT_PREP AS
SELECT p.payment_id, p.order_id, o.customer_id, o.order_date, p.paid_at, CAST(p.paid_at AS DATE) AS paid_date,
       p.payment_status, p.payment_amount, p.has_valid_order
FROM CORE.PAYMENTS p LEFT JOIN CORE.ORDERS o ON p.order_id = o.order_id;

In [ ]:
-- 확인: 판정 결과 분포 (기대값: 매출 인식 27,055 / NOT_COMPLETED 2,190 / DEPENDENCY_ERROR 284 /
--        AMOUNT_MISMATCH 141 / SUCCESS_PAYMENT_COUNT 101 / MISSING_ITEMS 31). 제외된 주문도 모두 "적재"됩니다.
SELECT COALESCE(exclusion_reason, '(매출 인식)') AS result, COUNT(*) AS orders, SUM(order_amount) AS order_amount
FROM CORE.ORDER_QUALITY GROUP BY 1 ORDER BY orders DESC;

In [ ]:
-- 매출로 측정되지 못한 각 이유를 보여주는 테이블
SELECT order_id, order_status, order_amount, line_amount_total, success_payment_count, exclusion_reason, is_revenue_eligible
FROM CORE.ORDER_QUALITY 
WHERE exclusion_reason IS NOT NULL ORDER BY order_id;

## STEP 6. 감사 기록 생성
1. 이번 배치의 이슈를 모아든 CORE.ISSUE_WORK 테이블에서 AUDIT.DATA_ISSUES 테이블로 옮깁니다.
2. RUN_SUMMARY 테이블을 통해 각 게이트별 통과한 행수를 측정합니다.

In [ ]:
INSERT INTO AUDIT.DATA_ISSUES (load_id, source_file, source_row_number, record_key, error_code, error_detail, order_id, action)
SELECT $load_id, source_file, source_row_number, record_key, error_code, error_detail, order_id, action
FROM CORE.ISSUES_WORK;

In [ ]:
-- RUN_SUMMARY 테이블
CREATE OR REPLACE TABLE CORE.RUN_SUMMARY AS
SELECT (SELECT COUNT(*) FROM RAW.V_CUSTOMERS_LATEST) + (SELECT COUNT(*) FROM RAW.V_PRODUCTS_LATEST)
     + (SELECT COUNT(*) FROM RAW.V_ORDERS_LATEST) + (SELECT COUNT(*) FROM RAW.V_ORDER_ITEMS_LATEST)
     + (SELECT COUNT(*) FROM RAW.V_PAYMENTS_LATEST)                                              AS input_rows,
       (SELECT COUNT(*) FROM CORE.ISSUES_WORK WHERE action = 'REMOVED')                          AS duplicate_rows,
       (SELECT COUNT(DISTINCT source_file || ':' || CAST(source_row_number AS VARCHAR))
          FROM CORE.ISSUES_WORK WHERE action = 'QUARANTINED')                                    AS quarantined_rows,
       (SELECT COUNT(*) FROM CORE.ISSUES_WORK WHERE action = 'FLAGGED')                          AS flagged_issues,
       (SELECT COUNT(*) FROM CORE.ORDER_QUALITY WHERE is_revenue_eligible)                       AS revenue_orders,
       (SELECT COALESCE(SUM(order_amount), 0) FROM CORE.ORDER_QUALITY WHERE is_revenue_eligible) AS revenue_amount;

SELECT * FROM CORE.RUN_SUMMARY;   -- 기대값: 129,618 / 473 / 441 / 559 / 27,055 / 2,198,198,700

## STEP 7. MART 반영

```
7-1 (SCD2 차원 적용) : 바뀐 고객/상품은 기존 버전을 닫고(valid_to) 새 버전을 추가
7-2 (Fact MERGE 적재) : 대리키는 "주문일 시점에 유효했던 버전"으로 연결 (point-in-time lookup)
                       없는 참조는 -1, 이번 스냅샷에서 사라진 행은 is_deleted = TRUE
7-3 품질 검사
7-4 게이트(SUCCESS/FAILED)
```

> **노트북 방식의 차이**: 노트북은 셀마다 따로 실행되므로 MART 반영을 하나의 트랜잭션 블록 대신 아래 셀로 나눴습니다. 각 셀의 MERGE/UPDATE 는 같은 `load_id` 로 다시 실행해도 결과가 같습니다(멱등). 대신 **검사 실패 시 자동 ROLLBACK 은 없습니다.** `FAILED` 가 나오면 원인을 고친 뒤 7-1a 부터 다시 실행하세요. 되돌려야 하면 `CREATE OR REPLACE TABLE … CLONE … AT(TIMESTAMP => <run_ts>)` 같은 Time Travel 을 쓰세요.

### 7-0. 실행 컨텍스트 복원 + 가드
`CORE.RUN_CTX` 의 `load_id`/`run_ts` 를 세션 변수로 불러오고, 반영할 정상 주문이 있는지 확인합니다.

In [ ]:
%%sql -r dataframe_10
-- 새 세션(노트북 재시작 등)이면 변수가 사라집니다. CORE.RUN_CTX 에서 다시 읽어 옵니다.
SET load_id = (SELECT load_id FROM CORE.RUN_CTX);
SET run_ts  = (SELECT run_ts  FROM CORE.RUN_CTX);
SELECT $load_id AS load_id, $run_ts AS run_ts;

In [ ]:
-- 이번 배치에 정상 주문이 있어야 MART 를 갱신합니다. 0 이면 아래 7-1 이후 셀을 실행하지 마세요.
SELECT $load_id AS load_id, $run_ts AS run_ts,
       (SELECT COUNT(*) FROM CORE.ORDERS) AS accepted_orders,
       IFF((SELECT COUNT(*) FROM CORE.ORDERS) = 0, '중단: 정상 주문 0건', '재개: 주문 건수 측정') AS guard;

### 7-1a. SCD2 : DIM_CUSTOMER
바뀐 고객은 기존 버전을 닫고(`valid_to`) 새 버전을 추가합니다. 같은 배치로 다시 실행해도 결과가 같습니다(멱등).

In [ ]:
%%sql -r dataframe_12
-- 7-1. SCD2 : dim_customer
UPDATE MART.DIM_CUSTOMER
   SET valid_to = $run_ts, is_current = FALSE
  FROM CORE.CUSTOMERS s
 WHERE MART.DIM_CUSTOMER.customer_id = s.customer_id
   AND MART.DIM_CUSTOMER.is_current
   AND MART.DIM_CUSTOMER.customer_key <> -1
   AND MART.DIM_CUSTOMER.hash_diff <> s.hash_diff;

INSERT INTO MART.DIM_CUSTOMER (customer_id, customer_name, email, region, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT s.customer_id, s.customer_name, s.email, s.region,
       CASE WHEN h.customer_id IS NULL THEN '1900-01-01'::TIMESTAMP_NTZ ELSE $run_ts END,  -- 처음 보는 고객은 과거 전체에 유효
       NULL, TRUE, s.hash_diff, $load_id
FROM CORE.CUSTOMERS s
LEFT JOIN (SELECT DISTINCT customer_id FROM MART.DIM_CUSTOMER) h ON h.customer_id = s.customer_id
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_CUSTOMER d WHERE d.customer_id = s.customer_id AND d.is_current);

### 7-1b. SCD2 : DIM_PRODUCT

In [ ]:
%%sql -r dataframe_13
-- 7-1. SCD2 : dim_product
UPDATE MART.DIM_PRODUCT
   SET valid_to = $run_ts, is_current = FALSE
  FROM CORE.PRODUCTS s
 WHERE MART.DIM_PRODUCT.product_id = s.product_id
   AND MART.DIM_PRODUCT.is_current
   AND MART.DIM_PRODUCT.product_key <> -1
   AND MART.DIM_PRODUCT.hash_diff <> s.hash_diff;

INSERT INTO MART.DIM_PRODUCT (product_id, product_name, category, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT s.product_id, s.product_name, s.category,
       CASE WHEN h.product_id IS NULL THEN '1900-01-01'::TIMESTAMP_NTZ ELSE $run_ts END,
       NULL, TRUE, s.hash_diff, $load_id
FROM CORE.PRODUCTS s
LEFT JOIN (SELECT DISTINCT product_id FROM MART.DIM_PRODUCT) h ON h.product_id = s.product_id
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_PRODUCT d WHERE d.product_id = s.product_id AND d.is_current);

### 7-2a. FACT_SALES_LINE (MERGE)
주문일 시점에 유효했던 차원 버전으로 대리키를 연결(point-in-time). 없는 참조는 -1, 사라진 행은 `is_deleted`.

In [ ]:
%%sql -r dataframe_14
-- 7-2. fact_sales_line
MERGE INTO MART.FACT_SALES_LINE AS t
USING (
  SELECT p.order_item_id, p.order_id,
         COALESCE(c.customer_key, -1)      AS customer_key,
         COALESCE(pr.product_key, -1)      AS product_key,
         COALESCE(d.date_key, -1)          AS order_date_key,
         COALESCE(s.order_status_key, -1)  AS order_status_key,
         p.order_date, p.quantity, p.unit_price, p.line_amount, p.is_revenue_eligible, p.exclusion_reason
  FROM CORE.SALES_LINE_PREP p
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = p.customer_id AND c.customer_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_PRODUCT pr
         ON pr.product_id = p.product_id AND pr.product_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= pr.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(pr.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d          ON d.calendar_date = p.order_date
  LEFT JOIN MART.DIM_ORDER_STATUS s  ON s.status_code = p.order_status
) AS src
ON t.order_item_id = src.order_item_id
WHEN MATCHED THEN UPDATE SET
  order_id = src.order_id, customer_key = src.customer_key, product_key = src.product_key,
  order_date_key = src.order_date_key, order_status_key = src.order_status_key, order_date = src.order_date,
  quantity = src.quantity, unit_price = src.unit_price, line_amount = src.line_amount,
  is_revenue_eligible = src.is_revenue_eligible, exclusion_reason = src.exclusion_reason,
  is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (order_item_id, order_id, customer_key, product_key, order_date_key, order_status_key, order_date,
   quantity, unit_price, line_amount, is_revenue_eligible, exclusion_reason, is_deleted, load_id, updated_at)
VALUES
  (src.order_item_id, src.order_id, src.customer_key, src.product_key, src.order_date_key, src.order_status_key, src.order_date,
   src.quantity, src.unit_price, src.line_amount, src.is_revenue_eligible, src.exclusion_reason, FALSE, $load_id, $run_ts);

UPDATE MART.FACT_SALES_LINE SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND order_item_id NOT IN (SELECT order_item_id FROM CORE.SALES_LINE_PREP);

### 7-2b. FACT_PAYMENT (MERGE)

In [ ]:
%%sql -r dataframe_15
-- 7-2. fact_payment
MERGE INTO MART.FACT_PAYMENT AS t
USING (
  SELECT p.payment_id, p.order_id,
         COALESCE(c.customer_key, -1)       AS customer_key,
         COALESCE(d.date_key, -1)           AS paid_date_key,
         COALESCE(s.payment_status_key, -1) AS payment_status_key,
         p.paid_at, p.payment_amount, p.has_valid_order
  FROM CORE.PAYMENT_PREP p
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = p.customer_id AND c.customer_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d            ON d.calendar_date = p.paid_date
  LEFT JOIN MART.DIM_PAYMENT_STATUS s  ON s.status_code = p.payment_status
) AS src
ON t.payment_id = src.payment_id
WHEN MATCHED THEN UPDATE SET
  order_id = src.order_id, customer_key = src.customer_key, paid_date_key = src.paid_date_key,
  payment_status_key = src.payment_status_key, paid_at = src.paid_at, payment_amount = src.payment_amount,
  has_valid_order = src.has_valid_order, is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (payment_id, order_id, customer_key, paid_date_key, payment_status_key, paid_at, payment_amount, has_valid_order, is_deleted, load_id, updated_at)
VALUES
  (src.payment_id, src.order_id, src.customer_key, src.paid_date_key, src.payment_status_key, src.paid_at, src.payment_amount,
   src.has_valid_order, FALSE, $load_id, $run_ts);

UPDATE MART.FACT_PAYMENT SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND payment_id NOT IN (SELECT payment_id FROM CORE.PAYMENT_PREP);

### 7-2c. FACT_ORDER_LIFECYCLE (MERGE)

In [ ]:
%%sql -r dataframe_16
-- 7-2. fact_order_lifecycle
MERGE INTO MART.FACT_ORDER_LIFECYCLE AS t
USING (
  SELECT q.order_id,
         COALESCE(c.customer_key, -1)      AS customer_key,
         COALESCE(d.date_key, -1)          AS order_date_key,
         COALESCE(pd.date_key, -1)         AS first_success_paid_date_key,
         COALESCE(s.order_status_key, -1)  AS order_status_key,
         q.order_date, q.first_success_paid_at, q.order_amount, q.line_count, q.line_amount_total,
         q.success_payment_count, q.success_payment_amount,
         DATEDIFF('day', q.order_date, CAST(q.first_success_paid_at AS DATE)) AS days_to_first_success_payment,
         q.is_revenue_eligible, q.exclusion_reason
  FROM CORE.ORDER_QUALITY q
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = q.customer_id AND c.customer_key <> -1
        AND CAST(q.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(q.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d          ON d.calendar_date  = q.order_date
  LEFT JOIN MART.DIM_DATE pd         ON pd.calendar_date = CAST(q.first_success_paid_at AS DATE)
  LEFT JOIN MART.DIM_ORDER_STATUS s  ON s.status_code    = q.order_status
) AS src
ON t.order_id = src.order_id
WHEN MATCHED THEN UPDATE SET
  customer_key = src.customer_key, order_date_key = src.order_date_key,
  first_success_paid_date_key = src.first_success_paid_date_key, order_status_key = src.order_status_key,
  order_date = src.order_date, first_success_paid_at = src.first_success_paid_at, order_amount = src.order_amount,
  line_count = src.line_count, line_amount_total = src.line_amount_total,
  success_payment_count = src.success_payment_count, success_payment_amount = src.success_payment_amount,
  days_to_first_success_payment = src.days_to_first_success_payment,
  is_revenue_eligible = src.is_revenue_eligible, exclusion_reason = src.exclusion_reason,
  is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (order_id, customer_key, order_date_key, first_success_paid_date_key, order_status_key, order_date, first_success_paid_at,
   order_amount, line_count, line_amount_total, success_payment_count, success_payment_amount, days_to_first_success_payment,
   is_revenue_eligible, exclusion_reason, is_deleted, load_id, updated_at)
VALUES
  (src.order_id, src.customer_key, src.order_date_key, src.first_success_paid_date_key, src.order_status_key, src.order_date,
   src.first_success_paid_at, src.order_amount, src.line_count, src.line_amount_total, src.success_payment_count,
   src.success_payment_amount, src.days_to_first_success_payment, src.is_revenue_eligible, src.exclusion_reason,
   FALSE, $load_id, $run_ts);

UPDATE MART.FACT_ORDER_LIFECYCLE SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND order_id NOT IN (SELECT order_id FROM CORE.ORDER_QUALITY);

### 7-3. 품질 검사 (17개)
`failures` 가 모두 0 이어야 합니다. 결과는 `AUDIT.DQ_RESULTS` 에 저장됩니다.

In [ ]:
%%sql -r dataframe_17
-- 7-3. 품질 검사 (failures 가 모두 0 이어야 COMMIT)
DELETE FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id;   -- 같은 배치를 다시 실행해도 결과가 중복되지 않게
INSERT INTO AUDIT.DQ_RESULTS (load_id, check_name, failures)
SELECT $load_id, check_name, failures FROM (
  SELECT 'dim_customer_one_current_version' AS check_name, COUNT(*) AS failures
    FROM (SELECT customer_id FROM MART.DIM_CUSTOMER GROUP BY customer_id HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1) x
  UNION ALL SELECT 'dim_customer_key_unique', COUNT(*)
    FROM (SELECT customer_key FROM MART.DIM_CUSTOMER GROUP BY customer_key HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'dim_customer_valid_range', COUNT(*)
    FROM MART.DIM_CUSTOMER WHERE valid_to IS NOT NULL AND valid_to <= valid_from
  UNION ALL SELECT 'dim_product_one_current_version', COUNT(*)
    FROM (SELECT product_id FROM MART.DIM_PRODUCT GROUP BY product_id HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1) x
  UNION ALL SELECT 'dim_product_key_unique', COUNT(*)
    FROM (SELECT product_key FROM MART.DIM_PRODUCT GROUP BY product_key HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_sales_line_key_unique', COUNT(*)
    FROM (SELECT order_item_id FROM MART.FACT_SALES_LINE GROUP BY order_item_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_sales_line_customer_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_CUSTOMER c ON f.customer_key = c.customer_key WHERE c.customer_key IS NULL
  UNION ALL SELECT 'fact_sales_line_product_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_PRODUCT p ON f.product_key = p.product_key WHERE p.product_key IS NULL
  UNION ALL SELECT 'fact_sales_line_date_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_DATE d ON f.order_date_key = d.date_key WHERE d.date_key IS NULL
  UNION ALL SELECT 'fact_sales_line_status_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_ORDER_STATUS s ON f.order_status_key = s.order_status_key WHERE s.order_status_key IS NULL
  UNION ALL SELECT 'fact_sales_line_amount_calc', COUNT(*)
    FROM MART.FACT_SALES_LINE WHERE quantity <= 0 OR unit_price < 0 OR line_amount <> quantity * unit_price
  UNION ALL SELECT 'fact_payment_key_unique', COUNT(*)
    FROM (SELECT payment_id FROM MART.FACT_PAYMENT GROUP BY payment_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_payment_status_fk', COUNT(*)
    FROM MART.FACT_PAYMENT f LEFT JOIN MART.DIM_PAYMENT_STATUS s ON f.payment_status_key = s.payment_status_key WHERE s.payment_status_key IS NULL
  UNION ALL SELECT 'fact_order_lifecycle_key_unique', COUNT(*)
    FROM (SELECT order_id FROM MART.FACT_ORDER_LIFECYCLE GROUP BY order_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'revenue_order_reconciliation', COUNT(*)
    FROM MART.FACT_ORDER_LIFECYCLE
   WHERE is_revenue_eligible AND NOT is_deleted
     AND (order_amount <> line_amount_total OR order_amount <> success_payment_amount OR success_payment_count <> 1)
  UNION ALL SELECT 'revenue_line_vs_order', COUNT(*)
    FROM MART.FACT_ORDER_LIFECYCLE o
    LEFT JOIN (SELECT order_id, SUM(line_amount) AS amt FROM MART.FACT_SALES_LINE WHERE NOT is_deleted GROUP BY order_id) l
           ON o.order_id = l.order_id
   WHERE o.is_revenue_eligible AND NOT o.is_deleted AND (l.amt IS NULL OR l.amt <> o.order_amount)
  UNION ALL SELECT 'revenue_requires_known_keys', COUNT(*)
    FROM MART.FACT_SALES_LINE
   WHERE is_revenue_eligible AND NOT is_deleted AND (customer_key = -1 OR product_key = -1 OR order_date_key = -1)
) checks;

In [ ]:
%%sql -r dataframe_18
SELECT check_name, failures FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id ORDER BY failures DESC, check_name;   -- 전부 0

### 7-4. 게이트 : 배치 상태 확정
검사 실패가 하나라도 있으면 `FAILED`, 모두 통과하면 `SUCCESS` 로 `AUDIT.PIPELINE_RUNS` 를 갱신합니다.

In [ ]:
%%sql -r dataframe_19
UPDATE AUDIT.PIPELINE_RUNS p
   SET status = IFF(q.fail_cnt = 0, 'SUCCESS', 'FAILED'),
       finished_at = CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
       message = IFF(q.fail_cnt = 0, NULL, 'DQ failed: ' || q.fail_names),
       input_rows = s.input_rows, duplicate_rows = s.duplicate_rows, quarantined_rows = s.quarantined_rows,
       flagged_issues = s.flagged_issues, revenue_orders = s.revenue_orders, revenue_amount = s.revenue_amount
  FROM CORE.RUN_SUMMARY s,
       (SELECT COALESCE(SUM(failures), 0) AS fail_cnt,
               COALESCE(LISTAGG(CASE WHEN failures > 0 THEN check_name END, ', '), '') AS fail_names
          FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id) q
 WHERE p.load_id = $load_id;

SELECT load_id, status, message, input_rows, revenue_orders, revenue_amount
FROM AUDIT.PIPELINE_RUNS WHERE load_id = $load_id;   -- 기대값: SUCCESS / 27,055 / 2,198,198,700

## STEP 8. 리포트 뷰 (Presentation layer)

```
- 모든 매출 뷰는 DIM_CUSTOMER 와 INNER JOIN 합니다.
     -> STEP 9 의 행 수준 보안이 DIM_CUSTOMER 에 걸리면, 뷰를 통한 매출도 자동으로 걸러집니다.
   - 분석가 역할에는 Fact 테이블이 아니라 이 뷰들만 권한을 줍니다.
```

In [ ]:
%%sql -r dataframe_20
-- >>> LOGIC START
CREATE OR REPLACE VIEW MART.V_REVENUE_DAILY AS
SELECT d.calendar_date AS order_date, d.day_name,
       COUNT(DISTINCT f.order_id) AS order_count,
       SUM(f.quantity)            AS quantity,
       SUM(f.line_amount)         AS revenue
FROM MART.FACT_SALES_LINE f
JOIN MART.DIM_DATE d     ON f.order_date_key = d.date_key
JOIN MART.DIM_CUSTOMER c ON f.customer_key = c.customer_key
WHERE f.is_revenue_eligible AND NOT f.is_deleted
GROUP BY d.calendar_date, d.day_name;

-- 전체 달력 활용: 매출이 없는 날도 0으로 보임
CREATE OR REPLACE VIEW MART.V_REVENUE_CALENDAR AS
WITH bounds AS (SELECT MIN(order_date) AS s, MAX(order_date) AS e FROM MART.FACT_ORDER_LIFECYCLE WHERE NOT is_deleted)
SELECT d.calendar_date, d.day_name, d.is_weekend,
       COALESCE(r.order_count, 0) AS order_count,
       COALESCE(r.revenue, 0)     AS revenue
FROM MART.DIM_DATE d
CROSS JOIN bounds b
LEFT JOIN MART.V_REVENUE_DAILY r ON r.order_date = d.calendar_date
WHERE d.calendar_date BETWEEN b.s AND b.e;

CREATE OR REPLACE VIEW MART.V_REVENUE_BY_PRODUCT AS
SELECT p.product_id, p.product_name, p.category,
       SUM(f.quantity) AS quantity, SUM(f.line_amount) AS revenue
FROM MART.FACT_SALES_LINE f
JOIN MART.DIM_PRODUCT p  ON f.product_key = p.product_key
JOIN MART.DIM_CUSTOMER c ON f.customer_key = c.customer_key
WHERE f.is_revenue_eligible AND NOT f.is_deleted
GROUP BY p.product_id, p.product_name, p.category;

-- SCD2 효과 1: 주문 "당시" 고객 지역 기준 (As-Was)
CREATE OR REPLACE VIEW MART.V_REVENUE_BY_REGION_ASWAS AS
SELECT c.region, COUNT(DISTINCT f.order_id) AS order_count, SUM(f.line_amount) AS revenue
FROM MART.FACT_SALES_LINE f
JOIN MART.DIM_CUSTOMER c ON f.customer_key = c.customer_key
WHERE f.is_revenue_eligible AND NOT f.is_deleted
GROUP BY c.region;

-- SCD2 효과 2: 고객의 "현재" 지역 기준 (As-Is)
CREATE OR REPLACE VIEW MART.V_REVENUE_BY_REGION_ASIS AS
SELECT cur.region, COUNT(DISTINCT f.order_id) AS order_count, SUM(f.line_amount) AS revenue
FROM MART.FACT_SALES_LINE f
JOIN MART.DIM_CUSTOMER c   ON f.customer_key = c.customer_key
JOIN MART.DIM_CUSTOMER cur ON cur.customer_id = c.customer_id AND cur.is_current
WHERE f.is_revenue_eligible AND NOT f.is_deleted
GROUP BY cur.region;

-- 매출에서 빠진 주문도 사유와 함께 조회 (기존 EXCLUDED_ORDERS 테이블 역할)
CREATE OR REPLACE VIEW MART.V_ORDER_QUALITY AS
SELECT o.order_id, o.order_date, s.status_name AS order_status, c.customer_id, c.region,
       o.order_amount, o.line_amount_total, o.success_payment_count, o.success_payment_amount,
       o.days_to_first_success_payment, o.is_revenue_eligible, o.exclusion_reason
FROM MART.FACT_ORDER_LIFECYCLE o
JOIN MART.DIM_ORDER_STATUS s ON o.order_status_key = s.order_status_key
JOIN MART.DIM_CUSTOMER c     ON o.customer_key = c.customer_key
WHERE NOT o.is_deleted;

CREATE OR REPLACE VIEW MART.V_PAYMENT_SUMMARY AS
SELECT s.status_name AS payment_status, COUNT(*) AS attempts, SUM(f.payment_amount) AS amount,
       SUM(CASE WHEN f.has_valid_order THEN 0 ELSE 1 END) AS orphan_payments
FROM MART.FACT_PAYMENT f
JOIN MART.DIM_PAYMENT_STATUS s ON f.payment_status_key = s.payment_status_key
JOIN MART.DIM_CUSTOMER c       ON f.customer_key = c.customer_key
WHERE NOT f.is_deleted
GROUP BY s.status_name;
-- 대시보드용 라인 단위 매출 : 기간·지역·카테고리 필터를 한 뷰에서 처리 (지역 = 주문 당시)
CREATE OR REPLACE VIEW MART.V_SALES_LINE AS
SELECT f.order_id, f.order_item_id, d.calendar_date AS order_date, d.year_month, d.is_weekend,
       c.customer_id, c.region, p.product_id, p.product_name, p.category,
       f.quantity, f.unit_price, f.line_amount AS revenue
FROM MART.FACT_SALES_LINE f
JOIN MART.DIM_DATE d     ON f.order_date_key = d.date_key
JOIN MART.DIM_CUSTOMER c ON f.customer_key = c.customer_key
JOIN MART.DIM_PRODUCT p  ON f.product_key = p.product_key
WHERE f.is_revenue_eligible AND NOT f.is_deleted;

-- 결제 시도 일별·지역별·상태별 (성공률 추이용)
CREATE OR REPLACE VIEW MART.V_PAYMENT_DAILY AS
SELECT d.calendar_date AS paid_date, c.region, s.status_name AS payment_status,
       COUNT(*) AS attempts, SUM(f.payment_amount) AS amount,
       SUM(CASE WHEN f.has_valid_order THEN 0 ELSE 1 END) AS orphan_payments
FROM MART.FACT_PAYMENT f
JOIN MART.DIM_DATE d           ON f.paid_date_key = d.date_key
JOIN MART.DIM_PAYMENT_STATUS s ON f.payment_status_key = s.payment_status_key
JOIN MART.DIM_CUSTOMER c       ON f.customer_key = c.customer_key
WHERE NOT f.is_deleted
GROUP BY d.calendar_date, c.region, s.status_name;
-- <<< LOGIC END (뷰)

CREATE OR REPLACE VIEW MART.V_LATEST_RUN AS
SELECT * FROM AUDIT.PIPELINE_RUNS QUALIFY ROW_NUMBER() OVER (ORDER BY started_at DESC) = 1;

-- 운영 뷰 : 대시보드가 AUDIT 테이블 권한 없이 실행 이력·품질 결과를 보도록 요약만 공개합니다 (개별 행·키 값 없음)
CREATE OR REPLACE VIEW MART.V_RUN_HISTORY AS
SELECT r.load_id, r.status, r.started_at, r.finished_at,
       DATEDIFF('second', r.started_at, r.finished_at) AS duration_sec,
       r.input_rows, r.duplicate_rows, r.quarantined_rows, r.flagged_issues,
       r.revenue_orders, r.revenue_amount, r.message,
       COALESCE(q.failed_checks, 0) AS failed_checks, COALESCE(q.total_checks, 0) AS total_checks
FROM AUDIT.PIPELINE_RUNS r
LEFT JOIN (SELECT load_id, SUM(CASE WHEN failures > 0 THEN 1 ELSE 0 END) AS failed_checks, COUNT(*) AS total_checks
           FROM AUDIT.DQ_RESULTS GROUP BY load_id) q ON q.load_id = r.load_id;

CREATE OR REPLACE VIEW MART.V_DQ_RESULTS AS
SELECT load_id, check_name, failures FROM AUDIT.DQ_RESULTS;

CREATE OR REPLACE VIEW MART.V_DATA_ISSUE_SUMMARY AS
SELECT load_id, action, error_code, source_file, COUNT(*) AS events
FROM AUDIT.DATA_ISSUES
GROUP BY load_id, action, error_code, source_file;


## STEP 9. 보안 (Enterprise Edition 이상)

```
9-1 역할 : AE101_ANALYST_SEOUL (서울 고객만, 이메일 마스킹) / AE101_MANAGER (전체 지역, 이메일 원문)
   9-2 Row Access Policy : DIM_CUSTOMER.region 기준. 역할-지역 매핑 테이블로 관리
   9-3 Masking Policy    : DIM_CUSTOMER.email
   다시 실행해도 안전하도록 "없으면 생성 -> 본문 갱신 -> 다시 연결" 순서로 작성했습니다.
```

In [ ]:
%%sql -r dataframe_25
select CURRENT_USER();

In [ ]:
%%sql -r dataframe_21
USE ROLE ACCOUNTADMIN;
SET my_user = CURRENT_USER();
SET my_wh   = CURRENT_WAREHOUSE();

-- 9-1. 역할과 권한
CREATE ROLE IF NOT EXISTS AE101_ANALYST_SEOUL;
CREATE ROLE IF NOT EXISTS AE101_MANAGER;
GRANT ROLE AE101_ANALYST_SEOUL TO ROLE SYSADMIN;
GRANT ROLE AE101_MANAGER       TO ROLE SYSADMIN;
GRANT ROLE AE101_ANALYST_SEOUL TO USER IDENTIFIER($my_user);   -- 실습용: 본인에게 두 역할을 부여
GRANT ROLE AE101_MANAGER       TO USER IDENTIFIER($my_user);


GRANT USAGE ON WAREHOUSE IDENTIFIER($my_wh)              TO ROLE AE101_ANALYST_SEOUL;
GRANT USAGE ON WAREHOUSE IDENTIFIER($my_wh)              TO ROLE AE101_MANAGER;
GRANT USAGE ON DATABASE AE101_SALES_ENT                  TO ROLE AE101_ANALYST_SEOUL;
GRANT USAGE ON DATABASE AE101_SALES_ENT                  TO ROLE AE101_MANAGER;
GRANT USAGE ON SCHEMA AE101_SALES_ENT.MART               TO ROLE AE101_ANALYST_SEOUL;
GRANT USAGE ON SCHEMA AE101_SALES_ENT.MART               TO ROLE AE101_MANAGER;
GRANT SELECT ON ALL VIEWS IN SCHEMA AE101_SALES_ENT.MART TO ROLE AE101_ANALYST_SEOUL;   -- Fact 테이블 권한은 주지 않음
GRANT SELECT ON ALL VIEWS IN SCHEMA AE101_SALES_ENT.MART TO ROLE AE101_MANAGER;
-- STEP 8 은 뷰를 CREATE OR REPLACE 로 다시 만들어 기존 권한이 사라집니다. 앞으로 만들 뷰에도 같은 권한을 줍니다.
GRANT SELECT ON FUTURE VIEWS IN SCHEMA AE101_SALES_ENT.MART TO ROLE AE101_ANALYST_SEOUL;
GRANT SELECT ON FUTURE VIEWS IN SCHEMA AE101_SALES_ENT.MART TO ROLE AE101_MANAGER;
GRANT SELECT ON TABLE AE101_SALES_ENT.MART.DIM_CUSTOMER  TO ROLE AE101_ANALYST_SEOUL;
GRANT SELECT ON TABLE AE101_SALES_ENT.MART.DIM_CUSTOMER  TO ROLE AE101_MANAGER;
GRANT SELECT ON TABLE AE101_SALES_ENT.MART.DIM_PRODUCT   TO ROLE AE101_ANALYST_SEOUL;
GRANT SELECT ON TABLE AE101_SALES_ENT.MART.DIM_PRODUCT   TO ROLE AE101_MANAGER;
GRANT SELECT ON TABLE AE101_SALES_ENT.MART.DIM_DATE      TO ROLE AE101_ANALYST_SEOUL;
GRANT SELECT ON TABLE AE101_SALES_ENT.MART.DIM_DATE      TO ROLE AE101_MANAGER;

-- 9-2. 역할-지역 매핑 ('*' = 전체 지역)
CREATE TABLE IF NOT EXISTS SECURITY.REGION_ACCESS_MAP (role_name VARCHAR, allowed_region VARCHAR);
DELETE FROM SECURITY.REGION_ACCESS_MAP;
INSERT INTO SECURITY.REGION_ACCESS_MAP (role_name, allowed_region) VALUES
  ('AE101_ANALYST_SEOUL', '서울'),
  ('AE101_MANAGER',       '*');

-- 9-2. Row Access Policy (인자 이름을 컬럼명과 다르게 둬야 서브쿼리에서 헷갈리지 않습니다)
CREATE ROW ACCESS POLICY IF NOT EXISTS SECURITY.RAP_CUSTOMER_REGION
  AS (cust_region VARCHAR) RETURNS BOOLEAN -> TRUE;

ALTER ROW ACCESS POLICY SECURITY.RAP_CUSTOMER_REGION SET BODY ->
  CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN')
  OR EXISTS (
    SELECT 1 FROM AE101_SALES_ENT.SECURITY.REGION_ACCESS_MAP m
    WHERE m.role_name = CURRENT_ROLE()
      AND (m.allowed_region = '*' OR m.allowed_region = cust_region));

ALTER TABLE MART.DIM_CUSTOMER DROP ALL ROW ACCESS POLICIES;
ALTER TABLE MART.DIM_CUSTOMER ADD ROW ACCESS POLICY SECURITY.RAP_CUSTOMER_REGION ON (region);

-- 9-3. Masking Policy : 허용 역할만 원문, 나머지는 '****@도메인'
CREATE MASKING POLICY IF NOT EXISTS SECURITY.MASK_EMAIL
  AS (val VARCHAR) RETURNS VARCHAR -> val;

ALTER MASKING POLICY SECURITY.MASK_EMAIL SET BODY ->
  CASE WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'AE101_MANAGER') THEN val
       WHEN val IS NULL THEN NULL
       ELSE REGEXP_REPLACE(val, '^[^@]+', '****') END;

ALTER TABLE MART.DIM_CUSTOMER MODIFY COLUMN email SET MASKING POLICY SECURITY.MASK_EMAIL FORCE;

-- 9-4. 체험 : 역할을 바꿔 가며 같은 쿼리를 실행해 보세요.
--   보조 역할(secondary roles)이 켜져 있으면 ACCOUNTADMIN 권한이 섞이므로, 실습 중에는 끕니다.
USE SECONDARY ROLES NONE;

USE ROLE AE101_ANALYST_SEOUL;
SELECT customer_id, customer_name, email, region FROM AE101_SALES_ENT.MART.DIM_CUSTOMER;  -- 서울 고객 389행만, email = ****@example.com
SELECT SUM(revenue) AS revenue FROM AE101_SALES_ENT.MART.V_REVENUE_DAILY;                -- 420,094,900 (서울 고객 주문만)
SELECT region, COUNT(*) AS orders FROM AE101_SALES_ENT.MART.V_ORDER_QUALITY GROUP BY region;   -- 서울 5,694건만 보임
-- SELECT * FROM AE101_SALES_ENT.MART.FACT_SALES_LINE;   -- 권한 없음 오류가 나야 정상

USE ROLE AE101_MANAGER;
SELECT customer_id, customer_name, email, region FROM AE101_SALES_ENT.MART.DIM_CUSTOMER;  -- 1,998행(Unknown 포함), email 원문
SELECT SUM(revenue) AS revenue FROM AE101_SALES_ENT.MART.V_REVENUE_DAILY;                -- 2,198,198,700

USE ROLE ACCOUNTADMIN;
USE SECONDARY ROLES ALL;

## STEP 10. 결과 확인 (ACCOUNTADMIN 기준)

In [ ]:
%%sql -r dataframe_22
SELECT * FROM MART.V_LATEST_RUN;                          -- status = SUCCESS

SELECT SUM(revenue) AS revenue, SUM(order_count) AS orders,
       SUM(revenue) / NULLIF(SUM(order_count), 0) AS avg_order_value
FROM MART.V_REVENUE_DAILY;                                -- 2,198,198,700 / 27,055 / 약 81,249

SELECT * FROM MART.V_REVENUE_CALENDAR ORDER BY calendar_date;
  -- 2025-01-01 ~ 2026-10-31, 669일. 매출이 없는 날이 있으면 0으로 표시됩니다.

-- 월별 매출 추이 (11~12월 성수기, 전년 대비 성장)
SELECT d.year_month, SUM(r.revenue) AS revenue, SUM(r.order_count) AS orders
FROM MART.V_REVENUE_DAILY r JOIN MART.DIM_DATE d ON r.order_date = d.calendar_date
GROUP BY d.year_month ORDER BY d.year_month;
  -- 2025-01 67,874,400 ... 2025-12 153,499,800 ... 2026-10 125,210,600

SELECT * FROM MART.V_REVENUE_BY_PRODUCT ORDER BY revenue DESC LIMIT 10;   -- 1위 프리미엄 키보드 199,525,700
SELECT category, SUM(revenue) AS revenue FROM MART.V_REVENUE_BY_PRODUCT GROUP BY category ORDER BY revenue DESC;
  -- 전자기기 921,160,700 / 생활용품 263,837,900 / 주방 247,517,500 ...
SELECT * FROM MART.V_REVENUE_BY_REGION_ASWAS ORDER BY revenue DESC;   -- 경기 635,003,900 / 서울 420,094,900 / 경남 169,902,600 ...
SELECT is_revenue_eligible, COUNT(*) FROM MART.V_ORDER_QUALITY GROUP BY 1;   -- TRUE 27,055 / FALSE 2,747 (합계 29,802)
SELECT * FROM MART.V_PAYMENT_SUMMARY ORDER BY payment_status;    -- 성공 27,702 / 실패 2,904 / 대기 360, 주문 없는 결제 108

-- 테이블 행 수 (기대값)
SELECT (SELECT COUNT(*) FROM MART.DIM_CUSTOMER)          AS dim_customer,         -- 1,998 (Unknown 포함)
       (SELECT COUNT(*) FROM MART.DIM_PRODUCT)           AS dim_product,          -- 151 (Unknown 포함)
       (SELECT COUNT(*) FROM MART.DIM_DATE)              AS dim_date,             -- 4,019
       (SELECT COUNT(*) FROM MART.FACT_SALES_LINE)       AS fact_sales_line,      -- 65,789
       (SELECT COUNT(*) FROM MART.FACT_PAYMENT)          AS fact_payment,         -- 30,966
       (SELECT COUNT(*) FROM MART.FACT_ORDER_LIFECYCLE)  AS fact_order_lifecycle; -- 29,802

-- 데이터 이슈: 무엇이 격리되고 무엇이 플래그만 됐는지
SELECT action, error_code, COUNT(*) AS events
FROM AUDIT.DATA_ISSUES WHERE load_id = $load_id GROUP BY action, error_code ORDER BY action, error_code;

SELECT check_name, failures FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id ORDER BY check_name;   -- 전부 0

## STEP 11. SCD2 체험 : 고객 C002 의 지역이 '부산'으로 바뀐 새 배치가 들어왔다고 가정

```
11-1 아래 4문장으로 새 실행 시작
   11-2 고객 파일만 변경된 새 배치를 RAW 에 추가 (다른 파일은 직전 배치를 그대로 씀)
   11-3 STEP 4 ~ STEP 8 을 다시 실행  (STEP 3 은 실행하지 않습니다)
   11-4 결과 비교
```

In [ ]:
%%sql -r dataframe_23
-- 11-1
-- UUID_STRING() 같은 비결정 함수는 SET 에 바로 쓸 수 없는 환경(노트북 등)이 있어, 먼저 테이블에 발급한 뒤 변수로 읽습니다.
CREATE OR REPLACE TABLE CORE.RUN_CTX AS
  SELECT UUID_STRING() AS load_id, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS run_ts;
SET load_id = (SELECT load_id FROM CORE.RUN_CTX);
SET run_ts  = (SELECT run_ts  FROM CORE.RUN_CTX);
INSERT INTO AUDIT.PIPELINE_RUNS (load_id, status, started_at) SELECT $load_id, 'RUNNING', $run_ts;

-- 11-2
INSERT INTO RAW.CUSTOMERS (load_id, loaded_at, src_file, src_row, customer_id, customer_name, email, region)
SELECT $load_id, $run_ts, src_file, src_row, customer_id, customer_name, email,
       CASE WHEN TRIM(customer_id) = 'C002' THEN '부산' ELSE region END
FROM RAW.V_CUSTOMERS_LATEST;

-- 11-3  >>> 여기서 STEP 4 ~ STEP 8 을 다시 실행하세요 <<<

-- 11-4
SELECT customer_key, customer_id, region, valid_from, valid_to, is_current
FROM MART.DIM_CUSTOMER WHERE customer_id = 'C002' ORDER BY valid_from;
  -- C002 가 2행: (Unknown, 1900-01-01 ~ 실행시각, FALSE) / (부산, 실행시각 ~ NULL, TRUE)

SELECT * FROM MART.V_REVENUE_BY_REGION_ASWAS WHERE region IN ('부산', 'Unknown');   -- 부산 120,881,700 / Unknown 51,063,400 (주문 당시 기준, 그대로)
SELECT * FROM MART.V_REVENUE_BY_REGION_ASIS  WHERE region IN ('부산', 'Unknown');   -- 부산 120,896,700 / Unknown 51,048,400 (O002 15,000원이 이동)

## STEP 12. 새롭게 시작 (리셋)

In [ ]:
%%sql -r dataframe_24
-- DROP DATABASE IF EXISTS AE101_SALES_ENT;
-- DROP ROLE IF EXISTS AE101_ANALYST_SEOUL;
-- DROP ROLE IF EXISTS AE101_MANAGER;